In [ ]:
"""
Temporal Test Set Construction and Analysis Pipeline
====================================================

This notebook constructs temporal holdout test sets from UniProt Gene Ontology
Annotation (GOA) files for protein function prediction benchmarking.

Pipeline Overview:
1. Download GAF files from UniProt for specified time periods
2. Filter annotations to experimental evidence only
3. Identify novel annotations in temporal window
4. Apply NK/LK (No Knowledge / Limited Knowledge) filtering
5. Propagate annotations through GO hierarchy
6. Format dataset for CAFA5 compatibility
7. Generate final Parquet file with HuggingFace metadata

Requirements:
- Clone repository and navigate to project directory
- Internet connection for downloading GAF files
- Python packages: pandas, numpy, datasets, cafaeval

Output:
- Filtered annotation files (TSV)
- Propagated GO term assignments (TSV)
- Final dataset with metadata (Parquet)
- Statistical summaries and protein lists
- Optional: HuggingFace dataset upload

Author
Ai Hub, UHN
November 2025

NOTE: Clone repository first. All required files will be downloaded automatically.
"""

In [ ]:
"""
Install Required Packages
-------------------------
Install necessary Python packages for GO annotation processing.
"""

import sys

# Install required packages
print("Installing required packages...")
!pip install -q datasets cafaeval

print("Package installation complete")

In [ ]:
"""
Import Libraries and Configure Logging
--------------------------------------
"""

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Set, Tuple, Dict
import logging
from datasets import load_dataset, Dataset
from cafaeval.parser import obo_parser
from cafaeval.graph import propagate
import warnings
import urllib.request
import subprocess
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

In [ ]:
"""
Configuration Parameters
------------------------
Set the GAF file versions to define the temporal window.

GAF file numbering corresponds to release dates:
- 212: November 17, 2022
- 214: March 15, 2023
- 217: September 21, 2023
- 219: February 9, 2024
- 223: October 20, 2024
- 226: May 3, 2025

Modify START_GAF and END_GAF to analyze different temporal periods.
"""

# Configure temporal window
START_GAF = 214  # Baseline (start of test period)
END_GAF = 226    # Endpoint (end of extended period)

# Constants
EXPERIMENTAL_EVIDENCE_CODES = ['IDA', 'IPI', 'EXP', 'IGI', 'IMP', 'IEP', 'IC', 'TAS']
GAF_COLUMNS = [
    'DB', 'DB_ID', 'DB_Symbol', 'Qualifier', 'GO_ID',
    'DB_Reference', 'Evidence_Code', 'With_From', 'Aspect',
    'DB_Name', 'DB_Synonym', 'DB_Type', 'Taxon', 'Date',
    'Assigned_By', 'Extension', 'Gene_Product_Form_ID'
]
ASPECT_NAMES = {'P': 'BP', 'F': 'MF', 'C': 'CC'}

# Create output directory name based on configuration
OUTPUT_DIR = Path(f"cafa5_temporal_holdout_{START_GAF}_{END_GAF}_Nov25")

print(f"Configuration:")
print(f"  Temporal window: GAF {START_GAF} -> GAF {END_GAF}")
print(f"  Output directory: {OUTPUT_DIR}")

In [ ]:
"""
Download GAF Files and GO Ontology
----------------------------------
Downloads UniProt GOA files and GO ontology from public repositories.
Files are downloaded based on the configuration specified above.
"""

def download_file(url: str, filename: str):
    """Download file if it doesn't already exist."""
    if os.path.exists(filename):
        print(f"  File already exists: {filename}")
        return

    print(f"  Downloading: {filename}")
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"  Download complete: {filename}")
    except Exception as e:
        print(f"  Error downloading {filename}: {e}")
        raise

print("Downloading required files...")
print()

# Download GAF files based on configuration
base_url = "https://ftp.ebi.ac.uk/pub/databases/GO/goa/UNIPROT"
download_file(
    f"{base_url}/goa_uniprot_all.gaf.{START_GAF}.gz",
    f"goa_uniprot_all.gaf.{START_GAF}.gz"
)
download_file(
    f"{base_url}/goa_uniprot_all.gaf.{END_GAF}.gz",
    f"goa_uniprot_all.gaf.{END_GAF}.gz"
)

# Download GO ontology
download_file(
    "http://purl.obolibrary.org/obo/go/go-basic.obo",
    "go-basic.obo"
)

print()
print("All files downloaded successfully")

In [ ]:
"""
Filter GAF Files to Experimental Annotations
--------------------------------------------
Filters GAF files to retain only:
- UniProtKB entries
- Experimental evidence codes (IDA, IPI, EXP, IGI, IMP, IEP, IC, TAS)
- Protein-level annotations
"""

def filter_gaf_file(input_file: str, output_file: str):
    """Filter GAF file using awk command."""
    if os.path.exists(output_file):
        print(f"  Filtered file already exists: {output_file}")
        return

    print(f"  Filtering {input_file}...")

    cmd = f"""
    pigz -dc {input_file} | awk -F'\\t' \
    '!/^!/ && $1=="UniProtKB" && $7~/^(IDA|IPI|EXP|IGI|IMP|IEP|IC|TAS)$/ && $12=="protein"' \
    > {output_file}
    """

    try:
        subprocess.run(cmd, shell=True, check=True)
        print(f"  Created: {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"  Error filtering {input_file}: {e}")
        raise

print("Filtering GAF files to experimental annotations...")
print()

# Filter both GAF files
filter_gaf_file(
    f"goa_uniprot_all.gaf.{START_GAF}.gz",
    f"filtered_goa_uniprot_exp-protein-only_{START_GAF}.gaf"
)

filter_gaf_file(
    f"goa_uniprot_all.gaf.{END_GAF}.gz",
    f"filtered_goa_uniprot_exp-protein-only_{END_GAF}.gaf"
)

print()
print("Filtering complete")
print()

# Display file information
print("Filtered file information:")
for gaf_num in [START_GAF, END_GAF]:
    filename = f"filtered_goa_uniprot_exp-protein-only_{gaf_num}.gaf"
    if os.path.exists(filename):
        size = os.path.getsize(filename) / (1024 * 1024)
        with open(filename) as f:
            lines = sum(1 for _ in f)
        print(f"  {filename}: {size:.1f} MB, {lines:,} annotations")

In [ ]:
"""
Core Functions: File Loading and Filtering
------------------------------------------
Functions for loading and processing GAF files.
"""

def load_gaf(filepath: Path) -> pd.DataFrame:
    """
    Load GAF file, skipping header comments.

    Args:
        filepath: Path to GAF file

    Returns:
        DataFrame with GAF annotations
    """
    logger.info(f"Loading {filepath.name}")
    df = pd.read_csv(
        filepath,
        sep='\t',
        comment='!',
        names=GAF_COLUMNS,
        header=None,
        low_memory=False
    )
    logger.info(f"Loaded {len(df):,} annotations")
    return df


def filter_experimental_annotations(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only annotations with experimental evidence codes.

    Args:
        df: DataFrame with annotations

    Returns:
        Filtered DataFrame with experimental annotations only
    """
    experimental = df[df['Evidence_Code'].isin(EXPERIMENTAL_EVIDENCE_CODES)].copy()
    logger.info(f"Filtered to {len(experimental):,} experimental annotations")
    return experimental


def create_annotation_key(df: pd.DataFrame) -> pd.Series:
    """
    Create unique identifier for each annotation.

    Args:
        df: DataFrame with annotations

    Returns:
        Series with unique keys (protein|GO_term|evidence)
    """
    return df['DB_ID'] + '|' + df['GO_ID'] + '|' + df['Evidence_Code']

print("File I/O functions defined")

In [ ]:
"""
Core Functions: Novel Annotation Detection
------------------------------------------
Identify annotations present in new release but not in baseline.
"""

def find_new_annotations(old_df: pd.DataFrame, new_df: pd.DataFrame,
                        output_dir: Path = None) -> pd.DataFrame:
    """
    Identify annotations present in new release but not in old.

    Args:
        old_df: Baseline annotations
        new_df: New release annotations
        output_dir: Directory for saving duplicate analysis (optional)

    Returns:
        DataFrame with novel annotations
    """
    old_df['key'] = create_annotation_key(old_df)
    new_df['key'] = create_annotation_key(new_df)

    old_keys = set(old_df['key'])
    new_keys = set(new_df['key'])
    novel_keys = new_keys - old_keys

    logger.info(f"Found {len(novel_keys):,} new unique annotations")

    novel_annotations = new_df[new_df['key'].isin(novel_keys)].copy()

    # Analyze duplicates if output directory specified
    if output_dir:
        duplicates = novel_annotations[novel_annotations.duplicated(subset=['key'], keep=False)]
        if len(duplicates) > 0:
            logger.info(f"Found {len(duplicates):,} duplicate records for {duplicates['key'].nunique():,} annotations")

            output_dir.mkdir(exist_ok=True)
            dup_file = output_dir / 'duplicate_annotations_analysis.tsv'
            duplicates_sorted = duplicates.sort_values(['key', 'Date'])
            duplicates_sorted.to_csv(dup_file, sep='\t', index=False)
            logger.info(f"Saved duplicate records to {dup_file}")

    novel_annotations.drop('key', axis=1, inplace=True)
    logger.info(f"Total new annotation records: {len(novel_annotations):,}")

    return novel_annotations

print("Novel annotation detection function defined")

In [ ]:
"""
Core Functions: CAFA5 Dataset Loading
-------------------------------------
Load CAFA5 training data for NK/LK filtering.
"""

def load_cafa5_train_proteins() -> pd.DataFrame:
    """
    Load CAFA5 training data with GO annotations.

    Returns:
        DataFrame with training proteins and annotations
    """
    logger.info("Loading CAFA5 training data")
    ds = load_dataset("wanglab/cafa5", name="cafa5_reasoning")
    train_df = ds['train'].to_pandas()
    logger.info(f"Loaded {len(train_df):,} training proteins")
    return train_df

print("CAFA5 data loading function defined")

In [ ]:
"""
Core Functions: NK/LK Filtering
-------------------------------
Filter test annotations based on training set knowledge.

Rules:
- NK (No Knowledge) proteins: Not in training, keep all annotations
- LK (Limited Knowledge) proteins: In training with null aspects,
  keep only annotations for previously null aspects
"""

def filter_by_training_knowledge(
    test_annotations: pd.DataFrame,
    train_proteins: pd.DataFrame,
    output_dir: Path
) -> pd.DataFrame:
    """
    Filter test annotations based on training set knowledge.

    Args:
        test_annotations: Annotations to filter
        train_proteins: Training set proteins with GO annotations
        output_dir: Directory for saving statistics

    Returns:
        Filtered DataFrame with NK/LK annotations
    """
    test_proteins = set(test_annotations['DB_ID'].unique())
    train_protein_ids = set(train_proteins['protein_id'].unique())

    # Categorize proteins
    nk_proteins = test_proteins - train_protein_ids
    lk_candidates = test_proteins & train_protein_ids

    logger.info(f"Test proteins total: {len(test_proteins):,}")
    logger.info(f"NK proteins (not in training): {len(nk_proteins):,}")
    logger.info(f"Proteins in both test and training: {len(lk_candidates):,}")

    # For LK candidates, identify null aspects in training
    aspect_mapping = {'P': 'go_bp', 'F': 'go_mf', 'C': 'go_cc'}
    protein_null_aspects = {}

    for protein_id in lk_candidates:
        train_row = train_proteins[train_proteins['protein_id'] == protein_id].iloc[0]
        null_aspects = []
        for aspect, go_col in aspect_mapping.items():
            if train_row[go_col] is None:
                null_aspects.append(aspect)

        if null_aspects:
            protein_null_aspects[protein_id] = null_aspects

    lk_proteins = set(protein_null_aspects.keys())
    logger.info(f"LK proteins (in training with null aspects): {len(lk_proteins):,}")

    # Filter annotations
    filtered_annotations = []
    stats = {
        'nk_annotations': 0,
        'lk_annotations': 0,
        'removed_annotations': 0,
        'aspect_breakdown': {'P': 0, 'F': 0, 'C': 0},
        'nk_proteins': set(),
        'lk_proteins': set(),
        'removed_proteins': set()
    }

    for _, ann in test_annotations.iterrows():
        protein_id = ann['DB_ID']
        aspect = ann['Aspect']

        if protein_id in nk_proteins:
            filtered_annotations.append(ann)
            stats['nk_annotations'] += 1
            stats['aspect_breakdown'][aspect] += 1
            stats['nk_proteins'].add(protein_id)
        elif protein_id in protein_null_aspects:
            if aspect in protein_null_aspects[protein_id]:
                filtered_annotations.append(ann)
                stats['lk_annotations'] += 1
                stats['aspect_breakdown'][aspect] += 1
                stats['lk_proteins'].add(protein_id)
            else:
                stats['removed_annotations'] += 1
                stats['removed_proteins'].add(protein_id)
        else:
            stats['removed_annotations'] += 1
            stats['removed_proteins'].add(protein_id)

    filtered_df = pd.DataFrame(filtered_annotations)

    # Save statistics
    output_dir.mkdir(exist_ok=True)
    stats_file = output_dir / 'nk_lk_filtering_stats.txt'
    with open(stats_file, 'w') as f:
        f.write("NK/LK Filtering Statistics\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Original annotations: {len(test_annotations):,}\n")
        f.write(f"Filtered annotations: {len(filtered_df):,}\n")
        f.write(f"Removed annotations: {stats['removed_annotations']:,}\n\n")

        f.write("Protein Categories:\n")
        f.write(f"  NK proteins: {len(nk_proteins):,}\n")
        f.write(f"  LK proteins: {len(lk_proteins):,}\n\n")

        f.write("Annotation Breakdown:\n")
        f.write(f"  NK annotations kept: {stats['nk_annotations']:,}\n")
        f.write(f"  LK annotations kept: {stats['lk_annotations']:,}\n\n")

        f.write("By Aspect:\n")
        for aspect, count in stats['aspect_breakdown'].items():
            f.write(f"  {ASPECT_NAMES[aspect]}: {count:,}\n")

    logger.info(f"Saved NK/LK filtering statistics to {stats_file}")
    return filtered_df

print("NK/LK filtering function defined")

In [ ]:
"""
Core Functions: GO Term Propagation
-----------------------------------
Propagate annotations through GO hierarchy using cafaeval.
"""

def propagate_annotations_cafaeval(annotations: pd.DataFrame, obo_file: Path) -> pd.DataFrame:
    """
    Propagate annotations using cafaeval's parser and propagation.

    Args:
        annotations: Annotations to propagate
        obo_file: Path to GO ontology OBO file

    Returns:
        DataFrame with propagated annotations
    """
    logger.info("Starting annotation propagation using cafaeval")

    all_input_proteins = set(annotations['DB_ID'].unique())
    logger.info(f"Input proteins: {len(all_input_proteins)}")

    # Parse OBO file
    ontologies = obo_parser(str(obo_file), valid_rel=("is_a", "part_of"))

    # Aspect mapping
    aspect_to_namespace = {
        'P': 'biological_process',
        'F': 'molecular_function',
        'C': 'cellular_component'
    }
    namespace_to_aspect = {v: k for k, v in aspect_to_namespace.items()}

    # Group annotations by namespace
    annotations_by_ns = {}
    for aspect, ns_name in aspect_to_namespace.items():
        ns_annotations = annotations[annotations['Aspect'] == aspect]
        if len(ns_annotations) > 0:
            annotations_by_ns[ns_name] = ns_annotations

    all_propagated = []

    for ns_name, ns_annotations in annotations_by_ns.items():
        if ns_name not in ontologies:
            logger.warning(f"Namespace {ns_name} not found in ontology")
            continue

        ont = ontologies[ns_name]
        unique_proteins = ns_annotations['DB_ID'].unique()
        protein_to_idx = {pid: i for i, pid in enumerate(unique_proteins)}

        # Create matrix for propagation
        matrix = np.zeros((len(unique_proteins), ont.idxs), dtype='bool')

        # Fill matrix with original annotations
        for _, row in ns_annotations.iterrows():
            go_id = row['GO_ID']
            protein_id = row['DB_ID']

            if go_id in ont.terms_dict:
                protein_idx = protein_to_idx[protein_id]
                term_idx = ont.terms_dict[go_id]['index']
                matrix[protein_idx, term_idx] = 1

        # Propagate
        propagate(matrix, ont, ont.order, mode='max')

        # Extract propagated annotations
        for protein_idx, protein_id in enumerate(unique_proteins):
            evidence_code = ns_annotations[ns_annotations['DB_ID'] == protein_id].iloc[0]['Evidence_Code']
            term_indices = np.where(matrix[protein_idx])[0]

            for term_idx in term_indices:
                go_id = None
                for term_id, term_info in ont.terms_dict.items():
                    if term_info['index'] == term_idx:
                        go_id = term_id
                        break

                if go_id:
                    template_row = ns_annotations[ns_annotations['DB_ID'] == protein_id].iloc[0].to_dict()
                    template_row['GO_ID'] = go_id
                    template_row['Aspect'] = namespace_to_aspect[ns_name]
                    all_propagated.append(template_row)

    # Convert to DataFrame and remove duplicates
    propagated_df = pd.DataFrame(all_propagated)
    propagated_df = propagated_df.drop_duplicates(
        subset=['DB_ID', 'GO_ID', 'Evidence_Code'],
        keep='first'
    )

    propagated_proteins = set(propagated_df['DB_ID'].unique())
    missing_proteins = all_input_proteins - propagated_proteins

    if missing_proteins:
        logger.warning(f"Lost {len(missing_proteins)} proteins during propagation: {missing_proteins}")

    logger.info(f"Output proteins: {len(propagated_proteins)}")
    logger.info(f"Propagation complete: {len(annotations):,} -> {len(propagated_df):,} annotations")

    return propagated_df

print("GO propagation function defined")

In [ ]:
"""
Utility Functions: Statistics and Summary Reports
-------------------------------------------------
Functions for generating statistics and summaries.
"""

def print_detailed_summary(novel_annotations: pd.DataFrame,
                          current_annotations: pd.DataFrame,
                          stage: str = ""):
    """
    Print detailed summary to console.

    Args:
        novel_annotations: Original novel annotations found
        current_annotations: Current state of annotations
        stage: Pipeline stage description
    """
    current_proteins = set(current_annotations['DB_ID'].unique())
    new_annotation_proteins = set(novel_annotations['DB_ID'].unique())

    aspect_counts = current_annotations['Aspect'].value_counts()
    total_annotations = len(current_annotations)

    # Proteins by aspect
    proteins_by_aspect = {}
    for aspect in ['P', 'F', 'C']:
        proteins_by_aspect[aspect] = set(
            current_annotations[current_annotations['Aspect'] == aspect]['DB_ID'].unique()
        )

    # Multi-aspect proteins
    all_three = proteins_by_aspect['P'] & proteins_by_aspect['F'] & proteins_by_aspect['C']
    bp_mf = (proteins_by_aspect['P'] & proteins_by_aspect['F']) - all_three
    bp_cc = (proteins_by_aspect['P'] & proteins_by_aspect['C']) - all_three
    mf_cc = (proteins_by_aspect['F'] & proteins_by_aspect['C']) - all_three

    evidence_counts = current_annotations['Evidence_Code'].value_counts()

    # Print summary
    print("\n" + "="*70)
    print(f"SUMMARY {stage}")
    print("="*70)

    print("\n1. PROTEIN COUNTS:")
    print(f"   Proteins with new annotations in temporal window: {len(new_annotation_proteins):,}")
    print(f"   Proteins at this stage: {len(current_proteins):,}")

    print(f"\n2. ANNOTATIONS AT THIS STAGE:")
    print(f"   Total annotations: {total_annotations:,}")
    if len(current_proteins) > 0:
        print(f"   Average annotations per protein: {total_annotations/len(current_proteins):.1f}")

    print("\n3. BREAKDOWN BY GO ASPECT:")
    for aspect in ['P', 'F', 'C']:
        if aspect in aspect_counts:
            count = aspect_counts[aspect]
            pct = count/total_annotations*100 if total_annotations > 0 else 0
            print(f"   {ASPECT_NAMES[aspect]}: {count:,} annotations ({pct:.1f}%)")

    print("\n4. UNIQUE PROTEINS PER ASPECT:")
    for aspect in ['P', 'F', 'C']:
        count = len(proteins_by_aspect.get(aspect, set()))
        print(f"   {ASPECT_NAMES[aspect]}: {count:,} unique proteins")

    print("\n5. PROTEINS WITH ANNOTATIONS IN MULTIPLE ASPECTS:")
    print(f"   All 3 aspects: {len(all_three):,}")
    print(f"   BP and MF only: {len(bp_mf):,}")
    print(f"   BP and CC only: {len(bp_cc):,}")
    print(f"   MF and CC only: {len(mf_cc):,}")

    print("\n6. EVIDENCE CODE DISTRIBUTION:")
    for code in EXPERIMENTAL_EVIDENCE_CODES:
        if code in evidence_counts.index:
            count = evidence_counts[code]
            pct = count/total_annotations*100 if total_annotations > 0 else 0
            print(f"   {code}: {count:,} ({pct:.1f}%)")

    print("\n" + "="*70)


def print_propagation_comparison(original: pd.DataFrame, propagated: pd.DataFrame):
    """
    Print comparison statistics between original and propagated annotations.

    Args:
        original: Original annotations before propagation
        propagated: Annotations after propagation
    """
    print("\n" + "="*70)
    print("ANNOTATION PROPAGATION COMPARISON")
    print("="*70)

    print("\n1. OVERALL STATISTICS:")
    print(f"   Original annotations: {len(original):,}")
    print(f"   Propagated annotations: {len(propagated):,}")
    print(f"   Increase factor: {len(propagated)/len(original):.2f}x")

    print("\n2. UNIQUE TERMS:")
    orig_terms = set(original['GO_ID'].unique())
    prop_terms = set(propagated['GO_ID'].unique())
    new_terms = prop_terms - orig_terms
    print(f"   Original unique GO terms: {len(orig_terms):,}")
    print(f"   Propagated unique GO terms: {len(prop_terms):,}")
    print(f"   New GO terms from propagation: {len(new_terms):,}")

    print("\n3. BY ASPECT COMPARISON:")
    for aspect in ['P', 'F', 'C']:
        orig_count = len(original[original['Aspect'] == aspect])
        prop_count = len(propagated[propagated['Aspect'] == aspect])
        if orig_count > 0:
            increase = (prop_count - orig_count) / orig_count * 100
            print(f"   {ASPECT_NAMES[aspect]}: {orig_count:,} -> {prop_count:,} (+{increase:.1f}%)")

    print("\n" + "="*70)

print("Utility functions defined")

In [ ]:
"""
Main Pipeline Execution - Part 1: Annotation Processing
-------------------------------------------------------
Execute the annotation extraction and propagation pipeline.
"""

def run_annotation_pipeline():
    """Execute annotation processing pipeline."""

    # Define file paths based on configuration
    old_gaf = Path(f"filtered_goa_uniprot_exp-protein-only_{START_GAF}.gaf")
    new_gaf = Path(f"filtered_goa_uniprot_exp-protein-only_{END_GAF}.gaf")
    obo_file = Path("go-basic.obo")

    logger.info("="*70)
    logger.info("TEMPORAL TEST SET CONSTRUCTION PIPELINE")
    logger.info("="*70)
    logger.info(f"Configuration: GAF {START_GAF} -> GAF {END_GAF}")
    logger.info(f"Output directory: {OUTPUT_DIR}")
    logger.info("="*70)

    # Load and filter GOA releases
    logger.info("Processing GOA releases")
    old_annotations = load_gaf(old_gaf)
    new_annotations = load_gaf(new_gaf)

    old_experimental = filter_experimental_annotations(old_annotations)
    new_experimental = filter_experimental_annotations(new_annotations)

    # Find new annotations
    novel_annotations = find_new_annotations(old_experimental, new_experimental, OUTPUT_DIR)

    # Use all proteins with new annotations
    test_annotations = novel_annotations.copy()
    logger.info(f"Using all {test_annotations['DB_ID'].nunique():,} proteins")

    # Print summary before NK/LK filtering
    print_detailed_summary(novel_annotations, test_annotations,
                          stage="BEFORE NK/LK FILTERING")

    # Save original annotations
    OUTPUT_DIR.mkdir(exist_ok=True)
    original_file = OUTPUT_DIR / 'cafa5_temporal_holdout_annotations_original.tsv'
    test_annotations.to_csv(original_file, sep='\t', index=False)
    logger.info(f"Saved original annotations to {original_file}")

    # Load training data for NK/LK filtering
    logger.info("Loading CAFA5 training data for NK/LK filtering")
    train_proteins = load_cafa5_train_proteins()

    # Apply NK/LK filtering
    nk_lk_annotations = filter_by_training_knowledge(
        test_annotations,
        train_proteins,
        OUTPUT_DIR
    )

    # Save NK/LK filtered annotations
    nk_lk_file = OUTPUT_DIR / 'cafa5_temporal_holdout_annotations_nk_lk.tsv'
    nk_lk_annotations.to_csv(nk_lk_file, sep='\t', index=False)
    logger.info(f"Saved NK/LK filtered annotations to {nk_lk_file}")

    # Print summary after NK/LK filtering
    print_detailed_summary(novel_annotations, nk_lk_annotations,
                          stage="AFTER NK/LK FILTERING (BEFORE PROPAGATION)")

    # Propagate annotations
    propagated_annotations = propagate_annotations_cafaeval(nk_lk_annotations, obo_file)

    # Print propagation comparison
    print_propagation_comparison(nk_lk_annotations, propagated_annotations)

    # Print final summary
    print_detailed_summary(novel_annotations, propagated_annotations,
                          stage="AFTER NK/LK FILTERING AND PROPAGATION")

    # Save final propagated results
    propagated_file = OUTPUT_DIR / 'cafa5_temporal_holdout_annotations_propagated.tsv'
    propagated_annotations.to_csv(propagated_file, sep='\t', index=False)
    logger.info(f"Saved propagated annotations to {propagated_file}")

    # Save final summary
    final_stats_file = OUTPUT_DIR / 'pipeline_summary.txt'
    with open(final_stats_file, 'w') as f:
        f.write("Temporal Test Set Construction Pipeline Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Configuration: GAF {START_GAF} -> GAF {END_GAF}\n\n")
        f.write("Files created:\n")
        f.write(f"1. {original_file.name} - Original annotations\n")
        f.write(f"2. {nk_lk_file.name} - NK/LK filtered annotations\n")
        f.write(f"3. {propagated_file.name} - Final propagated annotations\n\n")
        f.write("Annotation counts:\n")
        f.write(f"  Original: {len(test_annotations):,}\n")
        f.write(f"  After NK/LK filtering: {len(nk_lk_annotations):,}\n")
        f.write(f"  After propagation: {len(propagated_annotations):,}\n\n")
        f.write("Unique protein counts:\n")
        f.write(f"  Original: {test_annotations['DB_ID'].nunique():,}\n")
        f.write(f"  After NK/LK filtering: {nk_lk_annotations['DB_ID'].nunique():,}\n")
        f.write(f"  After propagation: {propagated_annotations['DB_ID'].nunique():,}\n")

    logger.info(f"Saved final summary to {final_stats_file}")

    print("\n" + "="*70)
    print("ANNOTATION PIPELINE COMPLETED")
    print("="*70)
    print(f"Results saved to: {OUTPUT_DIR}/")
    print("\nKey output files:")
    print(f"  - Original annotations: {original_file.name}")
    print(f"  - NK/LK filtered: {nk_lk_file.name}")
    print(f"  - Final propagated: {propagated_file.name}")
    print("="*70)

    return propagated_file

# Execute annotation pipeline
propagated_file = run_annotation_pipeline()

In [ ]:
"""
Dataset Formatting Functions
----------------------------
Convert propagated annotations to CAFA5 dataset format with metadata.
"""

def load_temporal_holdout_to_dataset_format(annotations_file: Path) -> pd.DataFrame:
    """
    Load temporal holdout annotations and convert to CAFA5 dataset format.

    Args:
        annotations_file: Path to the annotations TSV file

    Returns:
        DataFrame with columns matching CAFA5 dataset format
    """
    # Load annotations
    annotations = pd.read_csv(annotations_file, sep='\t')

    # Group by protein and aggregate GO terms by aspect
    protein_data = []

    for protein_id, group in annotations.groupby('DB_ID'):
        # Separate GO IDs by aspect
        bp_terms = group[group['Aspect'] == 'P']['GO_ID'].unique().tolist()
        mf_terms = group[group['Aspect'] == 'F']['GO_ID'].unique().tolist()
        cc_terms = group[group['Aspect'] == 'C']['GO_ID'].unique().tolist()

        # Get earliest date for each aspect
        bp_dates = group[group['Aspect'] == 'P']['Date'].tolist() if bp_terms else []
        mf_dates = group[group['Aspect'] == 'F']['Date'].tolist() if mf_terms else []
        cc_dates = group[group['Aspect'] == 'C']['Date'].tolist() if cc_terms else []

        # Find earliest date for each aspect
        bp_created = min(pd.to_datetime(bp_dates, format='%Y%m%d')) if bp_dates else None
        mf_created = min(pd.to_datetime(mf_dates, format='%Y%m%d')) if mf_dates else None
        cc_created = min(pd.to_datetime(cc_dates, format='%Y%m%d')) if cc_dates else None

        # Convert back to string format if not None
        bp_created = bp_created.strftime('%Y%m%d') if bp_created else None
        mf_created = mf_created.strftime('%Y%m%d') if mf_created else None
        cc_created = cc_created.strftime('%Y%m%d') if cc_created else None

        # Convert empty lists to None
        bp_terms = bp_terms if bp_terms else None
        mf_terms = mf_terms if mf_terms else None
        cc_terms = cc_terms if cc_terms else None

        # All GO IDs combined
        all_go_ids = []
        if bp_terms:
            all_go_ids.extend(bp_terms)
        if mf_terms:
            all_go_ids.extend(mf_terms)
        if cc_terms:
            all_go_ids.extend(cc_terms)

        all_go_ids = all_go_ids if all_go_ids else None

        # Create row in target format
        row = {
            'protein_id': protein_id,
            'protein_names': None,
            'protein_function': None,
            'organism': None,
            'length': None,
            'subcellular_location': None,
            'sequence': None,
            'go_ids': all_go_ids,
            'go_bp': bp_terms,
            'go_mf': mf_terms,
            'go_cc': cc_terms,
            'go_bp_created': bp_created,
            'go_mf_created': mf_created,
            'go_cc_created': cc_created,
            'interpro_ids': None,
            'structure_path': None,
            'string_id': None,
            'interaction_partners': None,
            'full_interaction_info': None
        }

        protein_data.append(row)

    # Convert to DataFrame
    df = pd.DataFrame(protein_data)
    df = df.sort_values('protein_id').reset_index(drop=True)

    print(f"Loaded {len(df)} proteins with temporal holdout annotations")
    print(f"Average GO terms per protein:")
    print(f"  - Total: {df['go_ids'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    print(f"  - BP: {df['go_bp'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    print(f"  - MF: {df['go_mf'].apply(lambda x: len(x) if x else 0).mean():.1f}")
    print(f"  - CC: {df['go_cc'].apply(lambda x: len(x) if x else 0).mean():.1f}")

    return df


def merge_temporal_holdout_with_hf_keep_all(annotations_file: Path) -> pd.DataFrame:
    """
    Merge temporal holdout GO annotations with HuggingFace metadata.
    Uses LEFT JOIN to keep ALL proteins.

    Args:
        annotations_file: Path to propagated annotations file

    Returns:
        Complete dataset with GO annotations and metadata
    """
    # Load HuggingFace test superset dataset
    print("Loading HuggingFace test_superset dataset...")
    test_superset_dataset = load_dataset("wanglab/cafa5", "cafa5_reasoning", split="test_superset")
    test_superset_df = test_superset_dataset.to_pandas()
    print(f"  HF test_superset has {len(test_superset_df)} proteins")

    # Load temporal holdout annotations
    print(f"\nLoading temporal holdout annotations from {annotations_file}...")
    df_temporal = load_temporal_holdout_to_dataset_format(annotations_file)
    print(f"  Temporal holdout has {len(df_temporal)} proteins")

    # Drop GO columns from HF dataset
    go_columns = ['go_ids', 'go_bp', 'go_mf', 'go_cc']
    hf_columns_to_keep = [col for col in test_superset_df.columns if col not in go_columns]
    test_superset_df = test_superset_df[hf_columns_to_keep]

    # LEFT JOIN - keep all temporal proteins
    print("\nMerging with HuggingFace metadata (LEFT JOIN)...")
    merged_df = df_temporal.merge(
        test_superset_df,
        on='protein_id',
        how='left',
        suffixes=('', '_hf')
    )

    # Update None values with HF data where available
    for col in test_superset_df.columns:
        if col != 'protein_id' and col in df_temporal.columns:
            hf_col = f'{col}_hf'
            if hf_col in merged_df.columns:
                merged_df[col] = merged_df[col].fillna(merged_df[hf_col])
                merged_df.drop(hf_col, axis=1, inplace=True)

    print(f"\nMerge complete:")
    print(f"  Total proteins: {len(merged_df)}")
    print(f"  Proteins with HF metadata: {merged_df['sequence'].notna().sum()}")
    print(f"  Proteins without HF metadata: {merged_df['sequence'].isna().sum()}")
    print(f"  Proteins with organism info: {merged_df['organism'].notna().sum()}")
    print(f"  Proteins with InterPro IDs: {merged_df['interpro_ids'].notna().sum()}")

    return merged_df

print("Dataset formatting functions defined")

In [ ]:
"""
Create Final Dataset with Metadata
----------------------------------
Generate final Parquet file ready for use or HuggingFace upload.
"""

print("="*70)
print("CREATING FINAL DATASET WITH METADATA")
print("="*70)

# Create dataset from propagated annotations
df_final = merge_temporal_holdout_with_hf_keep_all(propagated_file)

print("\n" + "="*70)
print("FINAL DATASET STATISTICS")
print("="*70)
print(f"Total proteins: {len(df_final)}")
print(f"\nGO annotation coverage:")
print(f"  Proteins with BP terms: {df_final['go_bp'].notna().sum()}")
print(f"  Proteins with MF terms: {df_final['go_mf'].notna().sum()}")
print(f"  Proteins with CC terms: {df_final['go_cc'].notna().sum()}")
print(f"\nMetadata coverage:")
print(f"  With sequences: {df_final['sequence'].notna().sum()} ({df_final['sequence'].notna().sum()/len(df_final)*100:.1f}%)")
print(f"  Without sequences: {df_final['sequence'].isna().sum()} ({df_final['sequence'].isna().sum()/len(df_final)*100:.1f}%)")
print(f"  With organism: {df_final['organism'].notna().sum()}")
print(f"  With InterPro: {df_final['interpro_ids'].notna().sum()}")

# Save the final dataset
output_parquet = f"cafa5_extended_test_set_ALL_PROTEINS_{START_GAF}_{END_GAF}.parquet"
df_final.to_parquet(output_parquet, index=False)
print(f"\nSaved to: {output_parquet}")
print("="*70)

# Display sample
print("\nSample (first 5 rows):")
print(df_final.head())